# PINN으로 파라볼라 안테나 전자기파 해석하기

물리 정보 신경망(PINN)은 인공지능이 물리 법칙을 지키면서 스스로 정답을 찾아가도록 가르치는 기술이다.  
일반 인공지능은 수많은 정답 데이터만 보고 학습하지만, PINN은 데이터가 없어도 "이 우주는 이런 물리 법칙(미분 방정식)을 따라야 해!"라는 규칙을 시험 문제집(손실 함수)에 직접 적어 알려주는 방식이다.  

여기서는 1m 크기의 **파라볼라 안테나(접시 모양 반사판)**와 전파를 쏴주는 **급전선(다이폴 안테나)**에서 전파가 어떻게 반사되어 뻗어나가는지 구현해본다.  


## 1. 입력 데이터 설명 (공간 도메인과 경계선)

인공지능 학습을 위해 두 가지 종류의 위치(좌표) 데이터를 준비해야 한다:  
1. **$X_u$ (경계선 데이터)**: 파라볼라 반사판 표면 위의 위치들이다. 반사판 표면에서는 전파가 부딪혀 0이 된다는 명확한 정답(경계 조건)을 이미 알고 있는 지점들이다.  
2. **$X_f$ (내부 자유 공간 데이터)**: 안테나 주변의 텅 빈 공간들이다. 여기서는 정답을 모르지만, 전자기 법칙(헬름홀츠 방정식)을 반드시 만족해야 하는 지점들이다.  

우리가 다룰 공간은 가로 $x \in [-0.6, 0.6]$ 미터, 세로 $y \in [0, 1.2]$ 미터 크기의 2차원 사각형이다. 안테나는 꼭짓점이 $(0,0)$이고 초점이 $(0,0.5)$인 포물선 $y = x^2 / 2$ 모양이다.  


### 아주 쉽게 이해하는 나이퀴스트 원리와 포락선(Envelope) 우회 비결

*   **나이퀴스트 원리란?**  
    바다의 파도가 아주 빠르게 칠 때, 사진을 가끔씩만 찍으면 파도가 멈춰있는 것처럼 보이거나 엉뚱한 모양으로 일그러진다(알리아싱 현상). 이처럼 빠르게 변하는 신호를 컴퓨터로 정밀하게 그리려면 엄청나게 많은 점을 조밀하게 찍어서 확인해야 한다.  
    주파수가 3 GHz인 고주파 전자기파는 공간상에서 매우 빠르게 진동한다. 일반적인 PINN으로 이를 계산하려면 수만 개의 점을 촘촘히 찍어야 해서 컴퓨터가 학습하기 전에 뻗어버린다.  

*   **어떻게 해결했는가? (포락선 기법)**  
    전파가 중앙에서 퍼져나가며 빠르게 파도치는 성분($e^{-ikr}$)은 우리가 이미 수학 공식으로 알고 있다. 그래서 이 빠르게 진동하는 성분을 수식으로 미리 걷어내고, 전체적인 파도의 높낮이 흐름인 **포락선(Envelope, $A(x,y)$)**만 인공지능이 그리도록 일을 시킨다.  
    포락선은 파도의 모양과 달리 완만하고 부드럽게 흐르기 때문에, 점을 많이 찍지 않아도(우리가 설정한 $N_f = 2000$개의 점만으로도) 나이퀴스트 조건을 위반하지 않고 매우 쉽고 정밀하게 학습할 수 있다.  

*   **SIREN 활성화 함수**:  
    인공지능이 둥근 안테나 경계면에서 튕겨 나가는 전파의 미세한 변화를 잘 그릴 수 있도록, 기존 활성화 함수 대신 사인 함수를 사용하여 미분 계산의 정밀도를 극대화한다.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
import time

# 재현성을 위한 난수 시드 고정
torch.manual_seed(42)
np.random.seed(42)

# ==========================================
# 1. 도메인 및 데이터 생성
# ==========================================

N_u = 200   # 파라볼라 경계면 위의 샘플 점 개수
N_f = 2000  # 자유 공간 내부의 샘플 점 개수

# 1-1. 파라볼라 반사판 경계점 생성 (꼭짓점 원점, F/D = 0.5 => 초점 F = 0.5)
# 포물선 공식: y = x^2 / (4F) = x^2 / 2 (x 범위는 -0.5에서 0.5까지)
x_bc = torch.linspace(-0.5, 0.5, N_u).view(-1, 1)
y_bc = (x_bc ** 2) / 2.0
X_u = torch.cat([x_bc, y_bc], dim=1)

# 1-2. 자유 공간 내부 검증점 생성 (반사판 위쪽 y > x^2/2 공간만 수집)
X_f_list = []
while len(X_f_list) < N_f:
    x_candidate = np.random.uniform(-0.6, 0.6)
    y_candidate = np.random.uniform(0.0, 1.2)
    # 파라볼라 반사판 위쪽 자유 공간에 위치하는 경우만 수집한다
    if y_candidate > (x_candidate ** 2) / 2.0:
        X_f_list.append([x_candidate, y_candidate])

X_f = torch.FloatTensor(X_f_list)

# 미분값을 추적하기 위해 requires_grad를 활성화한다
X_f.requires_grad_(True)

print(f"경계 조건 포인트 X_u 형태: {X_u.shape}")
print(f"내부 물리 포인트 X_f 형태: {X_f.shape}")

# 도메인 샘플 점 시각화
plt.figure(figsize=(7,7))
plt.scatter(X_f[:,0].detach().numpy(), X_f[:,1].detach().numpy(), c='lightgray', s=10, label='Free Space (X_f)')
plt.scatter(X_u[:,0].detach().numpy(), X_u[:,1].detach().numpy(), c='red', s=20, label='Parabolic Reflector (X_u)')
plt.scatter([0.0], [0.5], c='blue', marker='*', s=150, zorder=5, label='Dipole Feed Source (0, 0.5)')
plt.xlabel('x (m)')
plt.ylabel('y (m)')
plt.title('Collocation Points for Parabolic Antenna PINN')
plt.legend(loc='upper right')
plt.xlim(-0.7, 0.7)
plt.ylim(-0.1, 1.3)
plt.grid(True, alpha=0.3)
plt.show()


## 2. 하이퍼파라미터 설정 및 신경망 구조 시각화

물리 법칙이라는 까다로운 조건을 만족시키기 위해 총 5개의 은닉층과 은닉층당 64개의 노드로 신경망을 구성한다.  
또한, 빠르게 진동하는 전자기파의 경계면과 곡률을 잘 묘사하기 위해 사인 함수를 활성화 함수로 사용하는 **SIREN** 구조를 적용한다.  
신경망은 최종적으로 복소수 포락선의 실수부($A_r$)와 허수부($A_i$) 2가지를 예측해야 하므로 출력 크기는 **2**가 된다.  

In [ ]:
# ==========================================
# 2. 하이퍼파라미터 설정
# ==========================================

learning_rate = 1e-3 
epochs = 5000
hidden_layers = [64, 64, 64, 64, 64]

# 3 GHz 주파수 기준 물리 상수 설정
c_speed = 3e8         # 빛의 속도 (m/s)
freq = 3.0e9          # 3 GHz 동작 주파수
wavelength = c_speed / freq # 파장 lambda = 0.1 m
k_wavenumber = 2 * np.pi / wavelength # 파수 k = 20*pi (약 62.83 rad/m)

print(f"동작 주파수: {freq/1e9:.1f} GHz (wavelength = {wavelength:.2f} m)")
print(f"전자기 파수 k: {k_wavenumber:.4f} rad/m")
print(f"목표 에포크: {epochs}")
print(f"은닉층 구조: {hidden_layers}")
print(f"활성화 함수: SIREN (Sinusoidal Activation)")

def draw_neural_net(input_size, hidden_layers, output_size):
    """
    설정된 파라미터에 따라 신경망 형태를 화면에 대략적으로 그린다
    """
    vis_hidden = [min(h, 8) for h in hidden_layers]
    layer_sizes = [input_size] + vis_hidden + [output_size]
    
    fig = plt.figure(figsize=(10, 6))
    ax = fig.gca()
    ax.axis('off')
    
    left, right, bottom, top = 0.1, 0.9, 0.1, 0.9
    v_spacing = (top - bottom) / float(max(layer_sizes))
    h_spacing = (right - left) / float(len(layer_sizes) - 1)
    
    # 노드를 시각화한다
    for n, layer_size in enumerate(layer_sizes):
        layer_top = v_spacing * (layer_size - 1) / 2. + (top + bottom) / 2.
        for m in range(layer_size):
            circle = plt.Circle((n * h_spacing + left, layer_top - m * v_spacing), v_spacing / 5.,
                                color='w', ec='b', zorder=4)
            ax.add_artist(circle)
            # 중간 부분은 축약하여 표시한다
            if m == layer_size - 1 and hidden_layers[0] > 8 and n > 0 and n < len(layer_sizes)-1:
                ax.text(n * h_spacing + left, layer_top - (m+1.5) * v_spacing, '\n...\n(64)', ha='center', va='center', fontsize=10)
            
    # 노드 사이의 연결선을 그린다
    for n, (layer_size_a, layer_size_b) in enumerate(zip(layer_sizes[:-1], layer_sizes[1:])):
        layer_top_a = v_spacing * (layer_size_a - 1) / 2. + (top + bottom) / 2.
        layer_top_b = v_spacing * (layer_size_b - 1) / 2. + (top + bottom) / 2.
        for m in range(layer_size_a):
            for o in range(layer_size_b):
                line = plt.Line2D([n * h_spacing + left, (n + 1) * h_spacing + left],
                                  [layer_top_a - m * v_spacing, layer_top_b - o * v_spacing], c='k', alpha=0.1)
                ax.add_artist(line)
    
    plt.title(f"Parabolic PINN SIREN Architecture (Input: {input_size} -> {len(hidden_layers)} Layers -> Output: {output_size})")
    plt.show()

draw_neural_net(2, hidden_layers, 2)


## 3. 다이내믹 모델 생성 및 첫 순전파(Forward Pass) 시각화

설정한 조건에 맞춰 5층, 64노드의 단일(1자형) SIREN 모델을 생성한다.  
이 모델은 위치 $(x,y)$를 받아서 복소 포락선 $A = [A_r, A_i]$를 바로 내놓는다.  
학습되지 않은 초기 상태에서 임의의 위치에 대해 예측을 진행해보고, 은닉층을 통과할 때의 신호 흐름을 히트맵으로 관찰한다.  

In [ ]:
# ==========================================
# 3. 다이내믹 모델 생성 및 첫 순전파
# ==========================================

class SirenLayer(nn.Module):
    def __init__(self, in_features, out_features, is_first=False, omega_0=30.0):
        super(SirenLayer, self).__init__()
        self.omega_0 = omega_0
        self.is_first = is_first
        self.linear = nn.Linear(in_features, out_features)
        self.init_weights()
        
    def init_weights(self):
        # SIREN 논문에서 제시하는 가중치 초기화 규칙을 따른다
        with torch.no_grad():
            if self.is_first:
                self.linear.weight.uniform_(-1.0 / self.linear.in_features, 1.0 / self.linear.in_features)
            else: 
                bound = np.sqrt(6.0 / self.linear.in_features) / self.omega_0
                self.linear.weight.uniform_(-bound, bound)
            self.linear.bias.zero_()
            
    def forward(self, x):
        # 사인 활성화 함수를 가중치 배율 인자(omega_0)와 함께 사용한다
        return torch.sin(self.omega_0 * self.linear(x))

class DynamicSirenPINN(nn.Module):
    def __init__(self, input_size, hidden_layers, output_size, omega_0=30.0):
        super(DynamicSirenPINN, self).__init__()
        self.layers = nn.ModuleList()
        
        # 입력 레이어를 구성한다
        self.layers.append(SirenLayer(input_size, hidden_layers[0], is_first=True, omega_0=omega_0))
        
        # 은닉 레이어들을 구성한다
        for i in range(len(hidden_layers)-1):
            self.layers.append(SirenLayer(hidden_layers[i], hidden_layers[i+1], is_first=False, omega_0=omega_0))
            
        # 최종 출력 레이어는 선형(Linear)이다 (활성화 함수가 없다)
        self.output_layer = nn.Linear(hidden_layers[-1], output_size)
        with torch.no_grad():
            bound = np.sqrt(6.0 / hidden_layers[-1]) / omega_0
            self.output_layer.weight.uniform_(-bound, bound)
            self.output_layer.bias.zero_()
            
    def forward(self, x):
        activations = []
        out = x
        for layer in self.layers:
            out = layer(out)
            activations.append(out)
        out = self.output_layer(out)
        activations.append(out)
        # 최종 예측값과 각 은닉 레이어의 가중치 상태를 반환한다
        return out, activations

# 1자형 단일 네트워크 모델을 생성한다
model = DynamicSirenPINN(input_size=2, hidden_layers=hidden_layers, output_size=2, omega_0=30.0)

# 첫 순전파 수행 (시각화를 위해 X_f 중 상위 10개의 데이터 샘플만 사용한다)
sample_X = X_f[:10]
predictions, activations = model(sample_X)

print("[첫 순전파 결과 (상위 10개 샘플)]")
print("입력 좌표 X (x, y):\n", sample_X.detach().numpy()[:3], "...\n")
print("예측 복소 포락선 [A_r, A_i]:\n", predictions.detach().numpy()[:3], "...")

# 데이터 레이어 변화 시각화
fig, axes = plt.subplots(1, len(activations), figsize=(4 * len(activations), 4))
for i, (act, ax) in enumerate(zip(activations, axes)):
    cax = ax.matshow(act.detach().numpy(), cmap='viridis', aspect='auto')
    title = f"Layer {i+1} Output" if i < len(activations)-1 else "Final Output (Ar, Ai)"
    ax.set_title(title)
    ax.set_xlabel("Nodes" if i < len(activations)-1 else "Channels")
    if i == 0: ax.set_ylabel("Data Index")
    fig.colorbar(cax, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()


## 4. 오차(Loss) 계산 과정 시각화 (PINN의 핵심)

인공지능이 정답을 찾아가도록 유도하는 손실 함수(Loss)를 계산한다.  
PINN의 최종 손실 함수는 **경계 조건 오차(Loss_BC)**와 **물리 법칙 오차(Loss_PDE)**의 단순 합으로 정의된다.  

$$\text{Total Loss} = \text{Loss_BC} + \text{Loss_PDE}$$

1. **`Loss_BC` (경계 조건 손실)**:  
   안테나 표면($X_u$)에서는 입사된 전자기파가 반사되면서 완전히 상쇄되므로, 산란파의 포락선 세기는 반드시 $A_r = -1/\sqrt{r}$, $A_i = 0$이 되어야 한다. 이 정답값과 인공지능 예측값의 차이를 의미한다.  
2. **`Loss_PDE` (물리 법칙 손실)**:  
   자유 공간 내부($X_f$)에서는 복소 포락선이 아래의 전자기 파동 편미분 방정식을 완벽히 만족해야 한다:  
   $$\text{Residual } A_r = \nabla^2 A_r + 2k(\nabla A_i \cdot \nabla \phi) + k A_i \nabla^2 \phi = 0$$
   $$\text{Residual } A_i = \nabla^2 A_i - 2k(\nabla A_r \cdot \nabla \phi) - k A_r \nabla^2 \phi = 0$$
   이 공식이 0이 되지 않고 어긋나는 정도를 의미한다.  

PyTorch의 자동 미분 기능(`torch.autograd.grad`)을 사용하여 공간에 대한 포락선의 1차 미분 및 2차 미분 값을 계산해본다.  

In [ ]:
# ==========================================
# 4. 오차(Loss) 계산 과정 및 편미분 시각화
# ==========================================

# 계산 흐름을 보여주기 위해 샘플 5개만 임의로 추출한다
X_f_sample = X_f[:5]
u_pred_sample, _ = model(X_f_sample)

# 실수부와 허수부 예측값을 각각 떼어낸다
A_r_sample = u_pred_sample[:, 0:1]
A_i_sample = u_pred_sample[:, 1:2]

# 초점과의 거리 r과 위상 정보를 계산한다
x_s = X_f_sample[:, 0:1]
y_s = X_f_sample[:, 1:2]
eps = 1e-8
r_s = torch.sqrt(x_s**2 + (y_s - 0.5)**2 + eps)
phi_x_s = x_s / r_s
phi_y_s = (y_s - 0.5) / r_s
lap_phi_s = 1.0 / r_s

# 1. 자동 미분을 통한 1차 공간 미분
grad_A_r = torch.autograd.grad(A_r_sample, X_f_sample, grad_outputs=torch.ones_like(A_r_sample), create_graph=True)[0]
A_r_x = grad_A_r[:, 0:1]
A_r_y = grad_A_r[:, 1:2]

grad_A_i = torch.autograd.grad(A_i_sample, X_f_sample, grad_outputs=torch.ones_like(A_i_sample), create_graph=True)[0]
A_i_x = grad_A_i[:, 0:1]
A_i_y = grad_A_i[:, 1:2]

# 2. 자동 미분을 통한 2차 공간 미분
grad_A_r_x = torch.autograd.grad(A_r_x, X_f_sample, grad_outputs=torch.ones_like(A_r_x), create_graph=True)[0]
A_r_xx = grad_A_r_x[:, 0:1]
grad_A_r_y = torch.autograd.grad(A_r_y, X_f_sample, grad_outputs=torch.ones_like(A_r_y), create_graph=True)[0]
A_r_yy = grad_A_r_y[:, 1:2]
lap_A_r = A_r_xx + A_r_yy

grad_A_i_x = torch.autograd.grad(A_i_x, X_f_sample, grad_outputs=torch.ones_like(A_i_x), create_graph=True)[0]
A_i_xx = grad_A_i_x[:, 0:1]
grad_A_i_y = torch.autograd.grad(A_i_y, X_f_sample, grad_outputs=torch.ones_like(A_i_y), create_graph=True)[0]
A_i_yy = grad_A_i_y[:, 1:2]
lap_A_i = A_i_xx + A_i_yy

# 3. 포락선 전자기 방정식 오차(Loss_PDE) 계산
k = k_wavenumber
residual_r = lap_A_r + 2 * k * (A_i_x * phi_x_s + A_i_y * phi_y_s) + k * A_i_sample * lap_phi_s
residual_i = lap_A_i - 2 * k * (A_r_x * phi_x_s + A_r_y * phi_y_s) - k * A_r_sample * lap_phi_s
Loss_PDE_sample = torch.mean(residual_r**2 + residual_i**2)

print("--- 2차 미분 및 포락선 방정식 잔차 연산 추적 (첫 5개 샘플) ---")
print("1. A_r 예측값:\n", A_r_sample.detach().numpy())
print("2. A_r의 x축 방향 1차 미분값 (A_r_x):\n", A_r_x.detach().numpy())
print("3. A_r의 라플라시안 미분값 (lap_A_r):\n", lap_A_r.detach().numpy())
print("4. 실수부 물리 법칙 잔차:\n", residual_r.detach().numpy())
print(f"\n물리 법칙 만족 오차(Loss_PDE, 0에 가까울수록 정답): {Loss_PDE_sample.item():.6f}")

# 전체 학습에서 호출할 Loss_PDE 계산 함수를 정의한다
def calc_pde_loss_envelope(model_net, x_points, k_val):
    x = x_points[:, 0:1]
    y = x_points[:, 1:2]
    eps = 1e-8
    r = torch.sqrt(x**2 + (y - 0.5)**2 + eps)
    
    phi_x = x / r
    phi_y = (y - 0.5) / r
    lap_phi = 1.0 / r
    
    pred, _ = model_net(x_points)
    A_r = pred[:, 0:1]
    A_i = pred[:, 1:2]
    
    grad_A_r = torch.autograd.grad(A_r, x_points, grad_outputs=torch.ones_like(A_r), create_graph=True)[0]
    A_r_x = grad_A_r[:, 0:1]
    A_r_y = grad_A_r[:, 1:2]
    
    grad_A_i = torch.autograd.grad(A_i, x_points, grad_outputs=torch.ones_like(A_i), create_graph=True)[0]
    A_i_x = grad_A_i[:, 0:1]
    A_i_y = grad_A_i[:, 1:2]
    
    grad_A_r_x = torch.autograd.grad(A_r_x, x_points, grad_outputs=torch.ones_like(A_r_x), create_graph=True)[0]
    A_r_xx = grad_A_r_x[:, 0:1]
    grad_A_r_y = torch.autograd.grad(A_r_y, x_points, grad_outputs=torch.ones_like(A_r_y), create_graph=True)[0]
    A_r_yy = grad_A_r_y[:, 1:2]
    
    grad_A_i_x = torch.autograd.grad(A_i_x, x_points, grad_outputs=torch.ones_like(A_i_x), create_graph=True)[0]
    A_i_xx = grad_A_i_x[:, 0:1]
    grad_A_i_y = torch.autograd.grad(A_i_y, x_points, grad_outputs=torch.ones_like(A_i_y), create_graph=True)[0]
    A_i_yy = grad_A_i_y[:, 1:2]
    
    lap_A_r = A_r_xx + A_r_yy
    lap_A_i = A_i_xx + A_i_yy
    
    res_r = lap_A_r + 2 * k_val * (A_i_x * phi_x + A_i_y * phi_y) + k_val * A_i * lap_phi
    res_i = lap_A_i - 2 * k_val * (A_r_x * phi_x + A_r_y * phi_y) - k_val * A_r * lap_phi
    
    Loss_PDE = torch.mean(res_r**2 + res_i**2)
    return Loss_PDE


## 5. 역전파(Backpropagation) 및 가중치 업데이트 시각화

계산된 총 오차(Total Loss)를 모델 전체에 거꾸로 전달하여 각 연결 가중치가 오차를 줄이도록 갱신한다.  
경사하강법 및 최신 최적화 알고리즘인 `Adam`을 사용하여 가중치를 조절한다.  
학습 전과 후, 인공지능 모델의 첫 번째 층 연결 강도(가중치)가 어떻게 변화하는지 히트맵으로 대조해본다.  

In [ ]:
# ==========================================
# 5. 역전파 및 가중치 업데이트
# ==========================================

optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# 가중치를 업데이트하기 전 상태를 백업한다
old_weights = model.layers[0].linear.weight.data[:10, :].clone().numpy()

# 임시 타겟을 생성한다 (경계면에서 실수부는 -1.0, 허수부는 0.0을 목표로 한다)
u_pred_bc, _ = model(X_u)
target_bc_temp = torch.zeros_like(u_pred_bc)
target_bc_temp[:, 0] = -1.0

# 경계 조건 손실(Loss_BC)과 물리 법칙 손실(Loss_PDE)을 합쳐 전체 손실을 계산한다
Loss_BC = nn.MSELoss()(u_pred_bc, target_bc_temp)
Loss_PDE = calc_pde_loss_envelope(model, X_f, k_wavenumber)
total_loss = Loss_BC + Loss_PDE

# 역전파를 수행하여 가중치 조절 방향을 찾는다
optimizer.zero_grad() # 누적된 미분값 초기화
total_loss.backward() # 역전파 계산

# 기울기 값을 획득한다
gradients = model.layers[0].linear.weight.grad[:10, :].numpy()

# 가중치를 실질적으로 조절한다
optimizer.step()

# 업데이트된 후의 가중치를 획득한다
new_weights = model.layers[0].linear.weight.data[:10, :].numpy()

print(f"[1 에포크 학습 후의 Loss 결과]")
print(f"Loss_BC (경계 조건 오차): {Loss_BC.item():.6f}")
print(f"Loss_PDE (물리 법칙 오차): {Loss_PDE.item():.6f}")
print(f"Total Loss: {total_loss.item():.6f}\n")

# 가중치 변경 정보를 히트맵으로 그린다
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

def plot_heatmap(ax, data, title):
    cax = ax.matshow(data, cmap='coolwarm')
    for (row, col), val in np.ndenumerate(data):
        ax.text(col, row, f"{val:.3f}", ha='center', va='center', color='black', fontsize=9)
    ax.set_title(title)
    ax.set_xlabel("Input x, y")
    if title.startswith("1"):
        ax.set_ylabel("Hidden Nodes (Sample 10)")

plot_heatmap(axes[0], old_weights, "1. Old Weights (Before Update)")
plot_heatmap(axes[1], gradients, "2. Gradients (By Backprop)")
plot_heatmap(axes[2], new_weights, "3. New Weights (After Adam Update)")

plt.tight_layout()
plt.show()


## 6. 업데이트 된 결과와 에포크 간 오차 비교

가중치가 딱 1회 조절된 시점에서 오차가 실제로 얼마나 감소했는지 점검한다.  
경계 조건(Loss_BC)과 물리 법칙(Loss_PDE)은 서로 충돌할 수도 있어서, 학습 초반에는 두 오차의 균형을 맞추며 전체 오차가 서서히 줄어든다.  

In [ ]:
# ==========================================
# 6. 두 번째 순전파 및 오차 비교
# ==========================================

u_pred_bc_new, _ = model(X_u)

# 갱신된 가중치 상태로 두 종류의 손실 값을 다시 구한다
new_Loss_BC = nn.MSELoss()(u_pred_bc_new, target_bc_temp)
new_Loss_PDE = calc_pde_loss_envelope(model, X_f, k_wavenumber)
new_total_loss = new_Loss_BC + new_Loss_PDE

loss_epoch_1 = total_loss.item()
loss_epoch_2 = new_total_loss.item()

print(f"Epoch 1 전체 오차: {loss_epoch_1:.6f} (Loss_BC: {Loss_BC.item():.6f}, Loss_PDE: {Loss_PDE.item():.6f})")
print(f"Epoch 2 전체 오차: {loss_epoch_2:.6f} (Loss_BC: {new_Loss_BC.item():.6f}, Loss_PDE: {new_Loss_PDE.item():.6f})")
print(f"오차 감소량: {loss_epoch_1 - loss_epoch_2:.6f}\n")

# 막대 그래프로 오차 감소 수준을 보여준다
fig, ax = plt.subplots(figsize=(6, 4))
epochs_labels = ['Epoch 1', 'Epoch 2']
losses = [loss_epoch_1, loss_epoch_2]

bars = ax.bar(epochs_labels, losses, color=['coral', 'lightblue'], width=0.5)

for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, yval + (yval*0.01), f'{yval:.6f}', ha='center', va='bottom')

ax.set_ylabel('Total PINN Loss')
ax.set_title('Total Loss Comparison (Epoch 1 vs Epoch 2)')
ax.set_ylim(0, max(losses) * 1.2)
plt.show()


## 7. 본격적인 학습 1 : 포락선 물리 학습 (Envelope Learning)

사전 지식을 적용해 보았으니 설정된 반복 횟수(`epochs=5000`)에 걸쳐 전력 모델을 최종 학습한다.  
첫 번째 단계로 복소 포락선 $A(x,y)$를 학습하는 과정이다.  

*   **경계 조건 (Loss_BC)**: 반사판 표면($X_u$)에서 전파가 튕겨 나갈 때 복소 포락선 $A_r = -1/\sqrt{r}$, $A_i = 0$을 만족해야 한다.  
*   **물리 지배 방정식 (Loss_PDE)**: 자유 공간 내부($X_f$)에서 포락선이 헬름홀츠 방정식 잔차식을 충족해야 한다.  
학습을 다 마친 인공지능은 별도의 수식 연산 없이도 임의의 위치에 대한 복소 포락선 크기를 직접 근사해 낼 수 있다.  

In [ ]:
# ==========================================
# 7. 반복 학습 수행 및 결과 시각화 (포락선)
# ==========================================

# 본격적인 학습을 위해 완전히 깨끗한 새 모델을 초기화한다
model_envelope = DynamicSirenPINN(input_size=2, hidden_layers=hidden_layers, output_size=2, omega_0=30.0)
optimizer_envelope = optim.Adam(model_envelope.parameters(), lr=learning_rate)

# 경계면 타겟값을 정밀하게 계산해준다
x_bc_pts = X_u[:, 0:1]
y_bc_pts = X_u[:, 1:2]
r_bc = torch.sqrt(x_bc_pts**2 + (y_bc_pts - 0.5)**2)

target_bc = torch.zeros(N_u, 2)
target_bc[:, 0:1] = -1.0 / torch.sqrt(r_bc) # 실수 포락선 경계값
target_bc[:, 1:2] = 0.0 # 허수 포락선 경계값

loss_history_envelope = []
start_time = time.time()

print(f"--- 파라볼라 안테나 포락선 {epochs} 에포크 학습 시작 ---")
for epoch in range(epochs):
    optimizer_envelope.zero_grad()
    
    # 1. 경계면 오차 계산 (Loss_BC)
    u_pred_bc, _ = model_envelope(X_u)
    Loss_BC = nn.MSELoss()(u_pred_bc, target_bc)
    
    # 2. 자유 공간 내부의 물리 법칙 잔차 계산 (Loss_PDE)
    Loss_PDE = calc_pde_loss_envelope(model_envelope, X_f, k_wavenumber)
    
    # 3. 토탈 Loss 역전파 및 최적화
    loss = Loss_BC + Loss_PDE
    loss.backward()
    optimizer_envelope.step()
    
    loss_history_envelope.append(loss.item())
    
    # 10% 진행될 때마다 학습 로그를 출력한다
    if (epoch + 1) % (epochs // 10) == 0 or (epoch + 1) == 1:
        print(f"Epoch [{epoch+1:4d}/{epochs}], Total Loss: {loss.item():.6e} (Loss_BC: {Loss_BC.item():.6e}, Loss_PDE: {Loss_PDE.item():.6e})")

print(f"\n포락선 학습 완료! (소요 시간: {time.time() - start_time:.2f}초)")

# 학습 경과 그래프와 포락선 진폭 시각화 패널
fig = plt.figure(figsize=(15, 5))

# 1. Loss 곡선 시각화
ax1 = fig.add_subplot(1, 2, 1)
ax1.plot(loss_history_envelope, color='blue', linewidth=2)
ax1.set_yscale('log')
ax1.set_title('Envelope Training Loss Curve')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Total PINN Loss')
ax1.grid(True, linestyle='--', alpha=0.6)

# 2. 예측된 포락선의 절대적인 물리 크기 |A| = sqrt(Ar^2 + Ai^2) 히트맵
ax2 = fig.add_subplot(1, 2, 2)
x_grid = np.linspace(-0.6, 0.6, 150)
y_grid = np.linspace(0.0, 1.2, 150)
X_mesh, Y_mesh = np.meshgrid(x_grid, y_grid)
grid_tensor = torch.FloatTensor(np.c_[X_mesh.ravel(), Y_mesh.ravel()])

with torch.no_grad():
    A_pred, _ = model_envelope(grid_tensor)
A_pred = A_pred.numpy()
A_r_pred = A_pred[:, 0].reshape(X_mesh.shape)
A_i_pred = A_pred[:, 1].reshape(X_mesh.shape)
A_amp = np.sqrt(A_r_pred**2 + A_i_pred**2)

# 파라볼라 금속 반사판 내부 영역은 차단하여 표현하지 않는다
A_amp[Y_mesh < (X_mesh**2)/2.0] = np.nan

contour = ax2.contourf(X_mesh, Y_mesh, A_amp, levels=50, cmap='jet')
fig.colorbar(contour, ax=ax2)
ax2.set_title("Predicted Envelope Magnitude |A(x,y)|")
ax2.set_xlabel("x (m)")
ax2.set_ylabel("y (m)")
ax2.plot(x_grid, (x_grid**2)/2.0, 'k-', linewidth=2) # 안테나 물리 경계선을 그린다

plt.tight_layout()
plt.show()


## 8. 본격적인 학습 2 : 총 전계(Total Electric Field) 합성 및 해석

인공지능이 성공적으로 그려낸 포락선 $A(x,y)$에 미리 치워두었던 물결 모양의 위상 팩터 $e^{-ikr}$을 다시 결합하여 최종 반사된 전기장 $E_{sc}$를 만들어낸다.  
여기에 최초에 쏘아준 다이폴 입사 전기장 $E_{inc}$를 합하면 최종 전자기장 $E_{total}$이 합성된다.  

*   **입사파 (다이폴 피드)**: $E_{inc} = \frac{e^{-ikr}}{\sqrt{r}}$  
*   **산란파 (안테나 반사파)**: $E_{sc} = A(x,y) e^{-ikr}$  
*   **합성파 (최종 전자기장)**: $E_{total} = E_{inc} + E_{sc}$  

동그란 파도 형태의 입사파가 파라볼라 면에 닿아 곧고 뾰족하게 직진하는 평면파로 수집 및 발사되는 물리적 성질이 잘 나타나는지 확인한다.  

In [ ]:
# ==========================================
# 8. 총 전계 합성 및 결과 시각화
# ==========================================

r_grid = np.sqrt(X_mesh**2 + (Y_mesh - 0.5)**2 + 1e-8)
k = k_wavenumber

# 1. 다이폴 안테나 입사 전계의 실수부와 허수부를 계산한다
E_inc_r = np.cos(-k * r_grid) / np.sqrt(r_grid)
E_inc_i = np.sin(-k * r_grid) / np.sqrt(r_grid)

# 2. 복소수 포락선 예측 결과와 지수 위상 팩터를 결합하여 산란 전계를 구한다
# E_sc = A * e^{-ikr} 계산 공식을 코드로 풀어 쓴다
cos_kr = np.cos(-k * r_grid)
sin_kr = np.sin(-k * r_grid)
E_sc_r = A_r_pred * cos_kr - A_i_pred * sin_kr
E_sc_i = A_r_pred * sin_kr + A_i_pred * cos_kr

# 3. 총 전계를 합성한다
E_tot_r = E_inc_r + E_sc_r
E_tot_i = E_inc_i + E_sc_i
E_tot_amp = np.sqrt(E_tot_r**2 + E_tot_i**2)

# 안테나 금속면 아랫부분은 표현 영역에서 걸러낸다
mask = Y_mesh < (X_mesh**2)/2.0
E_tot_r[mask] = np.nan
E_tot_amp[mask] = np.nan

# 결과를 세 장의 사진 패널로 시각화한다
fig = plt.figure(figsize=(20, 5))

# 첫 번째 패널: 다이폴에서 사방으로 뻗어 나가는 원형 입사파
ax1 = fig.add_subplot(1, 3, 1)
E_inc_r[mask] = np.nan
contour_inc = ax1.contourf(X_mesh, Y_mesh, E_inc_r, levels=50, cmap='RdBu', vmin=-5, vmax=5)
fig.colorbar(contour_inc, ax=ax1)
ax1.plot(x_grid, (x_grid**2)/2.0, 'k-', linewidth=2)
ax1.scatter([0.0], [0.5], c='yellow', marker='*', s=150, zorder=5, edgecolors='black')
ax1.set_title("Incident Field Re{E_inc} (Dipole Feed)")
ax1.set_xlabel("x")
ax1.set_ylabel("y")

# 두 번째 패널: 안테나면에 부딪혀 평면파 형태로 위쪽으로 직진하는 합성 전계
ax2 = fig.add_subplot(1, 3, 2)
contour_tot = ax2.contourf(X_mesh, Y_mesh, E_tot_r, levels=50, cmap='RdBu', vmin=-5, vmax=5)
fig.colorbar(contour_tot, ax=ax2)
ax2.plot(x_grid, (x_grid**2)/2.0, 'k-', linewidth=2)
ax2.scatter([0.0], [0.5], c='yellow', marker='*', s=150, zorder=5, edgecolors='black')
ax2.set_title("Total Field Re{E_total} (Wave Focusing)")
ax2.set_xlabel("x")
ax2.set_ylabel("y")

# 세 번째 패널: 위로 곧게 뻗어 나가는 전자기 에너지 빔 크기 분포
ax3 = fig.add_subplot(1, 3, 3)
contour_amp = ax3.contourf(X_mesh, Y_mesh, E_tot_amp, levels=50, cmap='jet', vmax=6)
fig.colorbar(contour_amp, ax=ax3)
ax3.plot(x_grid, (x_grid**2)/2.0, 'k-', linewidth=2)
ax3.scatter([0.0], [0.5], c='yellow', marker='*', s=150, zorder=5, edgecolors='black')
ax3.set_title("Total Field Amplitude |E_total|")
ax3.set_xlabel("x")
ax3.set_ylabel("y")

plt.tight_layout()
plt.show()

# 안테나 금속면 위에서 도체 PEC 경계 오차(정밀도)를 점검한다
with torch.no_grad():
    A_bc_pred, _ = model_envelope(X_u)
A_bc_pred = A_bc_pred.numpy()

r_bc_np = r_bc.numpy()
cos_kr_bc = np.cos(-k * r_bc_np)
sin_kr_bc = np.sin(-k * r_bc_np)

E_sc_bc_r = A_bc_pred[:, 0:1] * cos_kr_bc - A_bc_pred[:, 1:2] * sin_kr_bc
E_sc_bc_i = A_bc_pred[:, 0:1] * sin_kr_bc + A_bc_pred[:, 1:2] * cos_kr_bc

E_inc_bc_r = np.cos(-k * r_bc_np) / np.sqrt(r_bc_np)
E_inc_bc_i = np.sin(-k * r_bc_np) / np.sqrt(r_bc_np)

E_tot_bc_r = E_inc_bc_r + E_sc_bc_r
E_tot_bc_i = E_inc_bc_i + E_sc_bc_i
pec_error = np.mean(E_tot_bc_r**2 + E_tot_bc_i**2) / np.mean(E_inc_bc_r**2 + E_inc_bc_i**2)

print(f"\n🎯 파라볼라 반사판 표면에서의 PEC 경계조건 오차율 (Loss_BC 관점): {pec_error * 100:.4f}%")
